# Install libs

In [1]:
# %pip install zss
# %pip install pandas

# Import packages and define variables

In [2]:
import os
import itertools
import functools
import pandas as pd
from zss import distance, simple_distance, Node
from zss.simple_tree import Node

# Get all graphs

In [3]:
# Define the input folder containing clustering solutions as .txt files
input_folder = "/home/leo/github/ils-clustering-hmd/data/clustering/metrics/input_joda_money"

def load_solution_entries(folder_path):
    """Load all .txt solutions in format: package_id;class_name"""
    solutions = {}

    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".txt"):
            continue

        solution_name = os.path.splitext(filename)[0]
        file_path = os.path.join(folder_path, filename)
        entries = []

        with open(file_path, "r", encoding="utf-8") as f:
            for line_number, raw_line in enumerate(f, 1):
                line = raw_line.strip()
                if not line:
                    continue

                parts = [x.strip() for x in line.split(";")]
                if len(parts) != 2:
                    raise ValueError(
                        f"Invalid line in {filename}:{line_number}. Expected 'package_id;class_name'"
                    )

                package_id, class_name = parts
                entries.append({
                    "package_id": package_id,
                    "class_name": class_name
                })

        solutions[solution_name] = entries

    return solutions

print("Loading .txt solutions...")
solutions = load_solution_entries(input_folder)
print(f"Loaded {len(solutions)} solution files")
print("Solutions:")
for name in sorted(solutions.keys()):
    print(f" - {name}: {len(solutions[name])} classes")

Loading .txt solutions...
Loaded 19 solution files
Solutions:
 - autores: 20 classes
 - otimizacao-CRG25: 20 classes
 - otimizacao-CRG50: 20 classes
 - otimizacao-CRG75: 20 classes
 - otimizacao-HMD: 20 classes
 - otimizacao-LNS: 20 classes
 - participante-ID01: 20 classes
 - participante-ID02: 20 classes
 - participante-ID03: 20 classes
 - participante-ID04: 20 classes
 - participante-ID05: 20 classes
 - participante-ID06: 20 classes
 - participante-ID07: 20 classes
 - participante-ID08: 20 classes
 - participante-ID09: 20 classes
 - participante-ID10: 20 classes
 - participante-ID11: 20 classes
 - participante-ID12: 20 classes
 - participante-ID13: 20 classes


In [4]:
# Create comparison pairs for every unordered pair of solution files (combinations)
print("\n=== Setting up comparison pairs (unordered) ===")

comparison_pairs = []
solution_names = sorted(solutions.keys())

for tree_a, tree_b in itertools.combinations(solution_names, 2):
    comparison_pairs.append({
        "tree_a": tree_a,
        "tree_b": tree_b,
        "entries_a": solutions[tree_a],
        "entries_b": solutions[tree_b]
    })

print(f"Created {len(comparison_pairs)} unordered comparison pairs")


=== Setting up comparison pairs (unordered) ===
Created 171 unordered comparison pairs


In [5]:
# Verify expected number of unordered pairs nC2 = n*(n-1)/2
print("=== Verification: Unordered All-vs-all Comparison Count ===")

n_files = len(solutions)
expected_total = n_files * (n_files - 1) // 2
actual_total = len(comparison_pairs)

print(f"Number of solution files: {n_files}")
print(f"Expected unordered comparisons (nC2): {n_files} * ({n_files} - 1) / 2 = {expected_total}")
print(f"Actual comparisons: {actual_total}")
print(f"✓ Match: {expected_total == actual_total}")

print("\nFiles included:")
for i, name in enumerate(sorted(solutions.keys()), 1):
    print(f"{i:2d}. {name}")

=== Verification: Unordered All-vs-all Comparison Count ===
Number of solution files: 19
Expected unordered comparisons (nC2): 19 * (19 - 1) / 2 = 171
Actual comparisons: 171
✓ Match: True

Files included:
 1. autores
 2. otimizacao-CRG25
 3. otimizacao-CRG50
 4. otimizacao-CRG75
 5. otimizacao-HMD
 6. otimizacao-LNS
 7. participante-ID01
 8. participante-ID02
 9. participante-ID03
10. participante-ID04
11. participante-ID05
12. participante-ID06
13. participante-ID07
14. participante-ID08
15. participante-ID09
16. participante-ID10
17. participante-ID11
18. participante-ID12
19. participante-ID13


# Define functions

In [6]:
def build_zss_tree_from_solution(solution_name, entries):
    """Build a ZSS tree where package hierarchy nodes contain class nodes as children."""
    root = Node(solution_name)

    package_nodes = {}
    for entry in entries:
        package_id = entry["package_id"]

        # Ensure all ancestors exist (e.g., create 1 before 1.1.2).
        parts = package_id.split(".")
        for i in range(1, len(parts) + 1):
            ancestor_id = ".".join(parts[:i])
            if ancestor_id not in package_nodes:
                package_nodes[ancestor_id] = Node(f"pkg:{ancestor_id}")

    # Connect package hierarchy to root.
    for package_id, package_node in package_nodes.items():
        if "." in package_id:
            parent_id = package_id.rsplit(".", 1)[0]
            package_nodes[parent_id].addkid(package_node)
        else:
            root.addkid(package_node)

    # Add classes to their package nodes.
    for entry in entries:
        package_id = entry["package_id"]
        class_name = entry["class_name"]
        class_node = Node(f"cls:{class_name}")
        package_nodes[package_id].addkid(class_node)

    return root

In [7]:
def zero_or_one_distance(a, b):
    if a == b:
        return 0
    else:
        return 1

def default_tree_size(tree,get_children):
    size = 1
    children = get_children(tree)
    if children:
        size = size + sum(default_tree_size(child,get_children) for child in children)
    return size

def _compute_normalized_distance(edit_distance,alpha,size_A,size_B):
    #return edit_distance / max(size_A, size_B)
    return edit_distance / (size_A + size_B)
    #return (2*edit_distance)/(alpha*(size_A+size_B)+edit_distance)

def normalized_simple_distance(A,B,get_children=Node.get_children,alpha=1,get_tree_size=None,return_operations=False,**kwargs):
    if get_tree_size is None:
        get_tree_size = functools.partial(default_tree_size,get_children=get_children)
    if return_operations:
        edit_distance, operations = simple_distance(A,B,get_children=get_children,**kwargs)
    else:
        edit_distance = simple_distance(A,B,get_children=get_children,**kwargs)
    d = _compute_normalized_distance(edit_distance,alpha,get_tree_size(A),get_tree_size(B))
    if return_operations:
        return d, operations
    else:
        return d

def normalized_distance(A,B,get_children,alpha=1,get_tree_size=None,return_operations=False,**kwargs):
    if get_tree_size is None:
        get_tree_size = functools.partial(default_tree_size,get_children=get_children)
    if return_operations:
        edit_distance, operations = distance(A,B,get_children,**kwargs)
    else:
        edit_distance = distance(A,B,get_children,**kwargs)
    d = _compute_normalized_distance(edit_distance,alpha,get_tree_size(A),get_tree_size(B))
    if return_operations:
        return d, operations
    else:
        return d

# Visual sanity check of generated trees

In [11]:
def tree_to_ascii(root):
    """Return an ASCII representation of a zss Node tree."""
    lines = [root.label]

    def walk(node, prefix=""):
        children = node.children
        for idx, child in enumerate(children):
            is_last = idx == len(children) - 1
            branch = "└── " if is_last else "├── "
            lines.append(prefix + branch + child.label)
            extension = "    " if is_last else "│   "
            walk(child, prefix + extension)

    walk(root)
    return "\n".join(lines)


def summarize_tree(root):
    """Quick structural summary for sanity checking."""
    stack = [(root, 0)]
    max_depth = 0
    package_count = 0
    class_count = 0

    while stack:
        node, depth = stack.pop()
        max_depth = max(max_depth, depth)

        if node.label.startswith("pkg:"):
            package_count += 1
        elif node.label.startswith("cls:"):
            class_count += 1

        for child in node.children:
            stack.append((child, depth + 1))

    return {
        "total_nodes": package_count + class_count + 1,  # + root
        "packages": package_count,
        "classes": class_count,
        "max_depth": max_depth,
    }


# Choose trees to inspect visually.
sample_tree_names = [
    "autores",
    "otimizacao-CRG25",
    "participante-ID01",
]

for tree_name in sample_tree_names:
    if tree_name not in solutions:
        print(f"Tree '{tree_name}' not found. Skipping.")
        continue

    tree = build_zss_tree_from_solution(tree_name, solutions[tree_name])
    stats = summarize_tree(tree)

    print("\n" + "=" * 90)
    print(f"TREE: {tree_name}")
    print(
        f"Summary -> total_nodes: {stats['total_nodes']}, "
        f"packages: {stats['packages']}, classes: {stats['classes']}, max_depth: {stats['max_depth']}"
    )
    print("=" * 90)
    print(tree_to_ascii(tree))


TREE: autores
Summary -> total_nodes: 23, packages: 2, classes: 20, max_depth: 3
autores
└── pkg:1
    ├── pkg:1.1
    │   ├── cls:AmountPrinterParser
    │   ├── cls:LiteralPrinterParser
    │   ├── cls:MoneyAmountStyle
    │   ├── cls:MoneyFormatException
    │   ├── cls:MoneyFormatter
    │   ├── cls:MoneyFormatterBuilder
    │   ├── cls:MoneyParseContext
    │   ├── cls:MoneyParser
    │   ├── cls:MoneyPrintContext
    │   └── cls:MoneyPrinter
    ├── cls:BigMoney
    ├── cls:BigMoneyProvider
    ├── cls:CurrencyMismatchException
    ├── cls:CurrencyUnit
    ├── cls:CurrencyUnitDataProvider
    ├── cls:DefaultCurrencyUnitDataProvider
    ├── cls:IllegalCurrencyException
    ├── cls:Money
    ├── cls:MoneyUtils
    └── cls:Ser

TREE: otimizacao-CRG25
Summary -> total_nodes: 27, packages: 6, classes: 20, max_depth: 4
otimizacao-CRG25
├── pkg:1
│   ├── cls:AmountPrinterParser
│   ├── cls:LiteralPrinterParser
│   ├── cls:MoneyAmountStyle
│   ├── cls:MoneyFormatterBuilder
│   ├── cls:M

# Calculate distance

In [8]:
# Calculate pairwise tree-edit distances
results = []

print(f"Starting distance calculations for {len(comparison_pairs)} comparison pairs...")

for i, pair in enumerate(comparison_pairs):
    tree_a = pair["tree_a"]
    tree_b = pair["tree_b"]
    entries_a = pair["entries_a"]
    entries_b = pair["entries_b"]

    if (i + 1) % 20 == 0:
        print(f"Progress: {i + 1}/{len(comparison_pairs)} comparisons completed")

    try:
        T1 = build_zss_tree_from_solution(tree_a, entries_a)
        T2 = build_zss_tree_from_solution(tree_b, entries_b)

        result, operations = simple_distance(T1, T2, return_operations=True)
        nresult = normalized_simple_distance(T1, T2)

        results.append({
            "tree_a": tree_a,
            "tree_b": tree_b,
            "distance": result,
            "normalized_distance": nresult,
            "operations": operations
        })

    except Exception as e:
        print(f"Error calculating distance for {tree_a} vs {tree_b}: {e}")
        results.append({
            "tree_a": tree_a,
            "tree_b": tree_b,
            "distance": None,
            "normalized_distance": None,
            "operations": None,
            "error": str(e)
        })

print("\nCompleted distance calculations!")
print(f"Total results: {len(results)}")
print(f"Successful calculations: {len([r for r in results if r.get('distance') is not None])}")
print(f"Failed calculations: {len([r for r in results if r.get('distance') is None])}")

Starting distance calculations for 171 comparison pairs...
Progress: 20/171 comparisons completed
Progress: 40/171 comparisons completed
Progress: 60/171 comparisons completed
Progress: 80/171 comparisons completed
Progress: 100/171 comparisons completed
Progress: 120/171 comparisons completed
Progress: 140/171 comparisons completed
Progress: 160/171 comparisons completed

Completed distance calculations!
Total results: 171
Successful calculations: 171
Failed calculations: 0


In [9]:
# Convert results to a DataFrame and save to CSV
results_df = pd.DataFrame(results)

# Remove operations column for the main results (too large for CSV)
if 'operations' in results_df.columns:
    results_df_clean = results_df.drop(columns=['operations'])
else:
    results_df_clean = results_df.copy()

# Save the DataFrame to a CSV file
results_df_clean.to_csv('results_joda_money.csv', index=False)


In [10]:
# Export operations data for detailed analysis
if results and any(r.get("operations") for r in results):
    first_with_ops = next((r for r in results if r.get("operations")), None)

    if first_with_ops:
        operations_df = pd.DataFrame(first_with_ops["operations"])
        operations_df["tree_a"] = first_with_ops["tree_a"]
        operations_df["tree_b"] = first_with_ops["tree_b"]

        operations_df.to_csv("operations_joda_money.csv", index=False)

        print("Tree edit operations saved to 'operations_joda_money.csv'")
        print(f"Operations from: {first_with_ops['tree_a']} vs {first_with_ops['tree_b']}")
        print(f"Number of operations: {len(operations_df)}")

        if "type" in operations_df.columns:
            print("\nOperation types:")
            print(operations_df["type"].value_counts())
    else:
        print("No operations data found in results")

    export_all = False  # Set to True if you want all operations data

    if export_all:
        all_operations = []
        for r in results:
            if r.get("operations"):
                for op in r["operations"]:
                    op_record = op.copy()
                    op_record["tree_a"] = r["tree_a"]
                    op_record["tree_b"] = r["tree_b"]
                    all_operations.append(op_record)

        if all_operations:
            all_ops_df = pd.DataFrame(all_operations)
            all_ops_df.to_csv("all_tree_edit_operations_joda_money.csv", index=False)
            print(f"All operations data saved to 'all_tree_edit_operations_joda_money.csv' ({len(all_operations)} total operations)")
else:
    print("No operations data available in results")

Tree edit operations saved to 'operations_joda_money.csv'
Operations from: autores vs otimizacao-CRG25
Number of operations: 35


- ref: https://zhang-shasha.readthedocs.io/en/latest/